Importing only required libraries, ensuring no reduntant or unused imports are loaded.

In [19]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.stats import mstats

Importing transaction_data (only used dataset). Data is importantly imported using a relative path to avoid issues with `FileNotFoundError`.

In [20]:
df = pd.read_csv('data/transaction_data.csv')

Data formatting, Assignment 1 Question 1

In [21]:
df_clean = df.rename(columns={
    'leadTime': 'lead_time',
    'Order-Quantity': 'order_quantity',
    'Manufacturing Site': 'manufacturing_site'
})

Data cleaning, Assignment 1 Question 2:

Removing transactions with negative `unit_price`.

Removing transactions with negative `order_quantity` (which are likely returns, not demand).

Removing the single large lead time outlier (2800).

In [22]:
df_clean = df_clean[df_clean['unit_price'] >= 0]
df_clean = df_clean[df_clean['order_quantity'] >= 0]
df_clean = df_clean[df_clean['lead_time'] < 2800]

Dealing with NaN Values, Assignment 1 Question 3:

Based on feedback, rows dropped where only essential identifiers are missing (`sku_number`, `inventory_type`, `stocking_type`).

This ensures data is retained is less important categories like `manufacturing_site`.

In [23]:
df_clean = df_clean.dropna(subset=['sku_number', 'inventory_type', 'stocking_type'])

print(f"Cleaned data shape: {df_clean.shape}")
print("\nDescription of Cleaned Numerical Data:")
display(df_clean.describe())

Cleaned data shape: (367271, 9)

Description of Cleaned Numerical Data:


,lead_time,unit_price,order_quantity
count,367271.000000,367271.000000,367271.000000
mean,26.258744,840.111016,84.796039
std,4.236911,343.717581,37.507969
min,14.000000,31.467547,0.000000
25%,28.000000,987.245293,72.000000
50%,28.000000,998.891440,94.000000
75%,28.000000,1006.701831,110.000000
max,28.000000,1023.638643,194.000000


Filtering for FG and MTS, setting up Assignment 1 Question 4:

Important step highlighted from the feedback on Assignment 1, analysis is made to be focused finished goods (FG) that are designated as make to stock (MTS). 

In [24]:
df_filtered = df_clean[
    (df_clean['inventory_type'] == 'FG') & 
    (df_clean['stocking_type'] == 'MTS')
].copy()

print(f"Filtered (FG, MTS) data shape: {df_filtered.shape}")
display(df_filtered.head())

Filtered (FG, MTS) data shape: (140394, 9)


,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
1,F6D696A7,FG,MTS,28,1011.724796,NaN,2056,2023-01-01,95
2,53B542CB,FG,MTS,28,1005.158694,PL-5C,06CC,2023-01-01,103
3,FE55EA7C,FG,MTS,28,997.138166,NaN,02B5,2023-01-01,58
6,3D7DF103,FG,MTS,28,996.426246,PL-F5,3B87,2023-01-01,104
7,4C2A4145,FG,MTS,28,986.911406,NaN,3FBB,2023-01-01,90


Unique Counts, Assignment 1 Question 4 (Part 1):

Below we use the properly filtered `df_filtered` DataFrame to find unique counts of SKUs, sites, and divisions.

In [25]:
unique_skus = df_filtered['sku_number'].nunique()
unique_sites = df_filtered['manufacturing_site'].nunique()
unique_divisions = df_filtered['division_code'].nunique()

print(f"Unique SKUs: {unique_skus}")
print(f"Unique Manufacturing Sites: {unique_sites}")
print(f"Unique Divisions: {unique_divisions}")

Unique SKUs: 282
Unique Manufacturing Sites: 15
Unique Divisions: 66


Top and Bottom Transactions by Quantity, Assignment 1 Question 4 (Part 2):

Orders the second part of Question 4, finding the top and bottom 10 transactions by `order_quantity`.

In [26]:
print("--- Top 10 by Order Quantity ---")
display(df_filtered.nlargest(10, 'order_quantity')[['sku_number', 'order_quantity']])

print("\n--- Bottom 10 by Order Quantity ---")
display(df_filtered.nsmallest(10, 'order_quantity')[['sku_number', 'order_quantity']])

--- Top 10 by Order Quantity ---


,sku_number,order_quantity
330858,F9D1D1FE,187
351878,32843A68,182
305472,0336A033,181
56975,0019425F,180
254637,7D60C5D8,180
258579,E6E01400,180
276850,633E44A4,180
354321,EC7A7615,178
40144,F6D696A7,177
302634,F6D696A7,177



--- Bottom 10 by Order Quantity ---


,sku_number,order_quantity
578,7D60C5D8,0
786,D6998D9A,0
957,BD57C63F,0
1468,D6998D9A,0
2964,8E83E81A,0
3302,5DD07FA1,0
3458,B8BD7009,0
3958,5F62CB7B,0
4150,60F2B655,0
4461,2FF2406B,0


The final part of Assignment 1 Question 4 is below:

This creates the `total_sales` column and get the top and bottom 10 transactions from this value.

In [27]:
df_filtered['total_sales'] = df_filtered['unit_price'] * df_filtered['order_quantity']

print("\n--- Top 10 by Total Sales ---")
display(df_filtered.nlargest(10, 'total_sales')[['sku_number', 'total_sales']])

print("\n--- Bottom 10 by Total Sales ---")
display(df_filtered.nsmallest(10, 'total_sales')[['sku_number', 'total_sales']])


--- Top 10 by Total Sales ---


,sku_number,total_sales
330858,F9D1D1FE,187703.960210
351878,32843A68,183278.577584
258579,E6E01400,182807.436454
305472,0336A033,180859.641232
56975,0019425F,180785.971545
276850,633E44A4,180310.237116
254637,7D60C5D8,179491.095217
40144,F6D696A7,179075.288887
302634,F6D696A7,179075.288887
96193,41608877,178557.992313



--- Bottom 10 by Total Sales ---


,sku_number,total_sales
578,7D60C5D8,0.0
786,D6998D9A,0.0
957,BD57C63F,0.0
1468,D6998D9A,0.0
2964,8E83E81A,0.0
3302,5DD07FA1,0.0
3458,B8BD7009,0.0
3958,5F62CB7B,0.0
4150,60F2B655,0.0
4461,2FF2406B,0.0


There were obvious problems in Assignment 1, and with it being my first true experience with Python, this is not really a surprise. Looking at the mistakes I made in it would be laughable today, but that goes to show how much I've learned at the same time. That's what it is all about, right?

Below will be the Questions for Assignment 2.

Assignment 2 Question 1 is featured below, aggregating filtered data by `sku_number` to get certain key demand statistics (min, max, mean, median, var, std) and average lead time for every SKU.

In [28]:
df_agg = df_filtered.groupby('sku_number').agg(
    qty_min=('order_quantity', 'min'),
    qty_max=('order_quantity', 'max'),
    qty_avg=('order_quantity', 'mean'),
    qty_median=('order_quantity', 'median'),
    qty_var=('order_quantity', 'var'),
    qty_std=('order_quantity', 'std'),
    avg_lead_time=('lead_time', 'mean')
).reset_index()

df_agg['qty_std'] = df_agg['qty_std'].fillna(0)
df_agg['qty_var'] = df_agg['qty_var'].fillna(0)

print("Aggregated SKU Statistics (Head):")
display(df_agg.head())

Aggregated SKU Statistics (Head):


,sku_number,qty_min,qty_max,qty_avg,qty_median,qty_var,qty_std,avg_lead_time
0,0019425F,38,180,98.726005,98.0,372.534060,19.301141,28.0
1,00DCA10C,0,28,9.834437,9.0,42.245740,6.499672,14.0
2,01FE860A,1,27,9.590909,9.0,41.689218,6.456719,14.0
3,0336A033,40,181,100.386293,100.0,418.525642,20.457899,28.0
4,058CD6C0,34,174,98.641903,98.0,406.347015,20.158051,28.0


Assignment 2 Question 2 had desired service levels be defined, using `scipy.stats.norm.ppf` to find the corresponding z-score for each of them.

In [29]:
service_levels = {
    '75%': 0.75,
    '90%': 0.90,
    '95%': 0.95
}

z_scores = {sl_name: norm.ppf(sl_prob) for sl_name, sl_prob in service_levels.items()}

print(f"Z-score for 75% SL: {z_scores['75%']:.4f}")
print(f"Z-score for 90% SL: {z_scores['90%']:.4f}")
print(f"Z-score for 95% SL: {z_scores['95%']:.4f}")

Z-score for 75% SL: 0.6745
Z-score for 90% SL: 1.2816
Z-score for 95% SL: 1.6449


Assigment 2 Question 3 applied the parametric safety stock formula for all three service levels, rounding the final result up to closest whole number.

In [30]:
df_ss = df_agg.copy()

df_ss['sqrt_lead_time'] = np.sqrt(df_ss['avg_lead_time'])

df_ss['ss_75_raw'] = z_scores['75%'] * df_ss['qty_std'] * df_ss['sqrt_lead_time']
df_ss['ss_90_raw'] = z_scores['90%'] * df_ss['qty_std'] * df_ss['sqrt_lead_time']
df_ss['ss_95_raw'] = z_scores['95%'] * df_ss['qty_std'] * df_ss['sqrt_lead_time']

df_ss['ss_75'] = np.ceil(df_ss['ss_75_raw']).fillna(0)
df_ss['ss_90'] = np.ceil(df_ss['ss_90_raw']).fillna(0)
df_ss['ss_95'] = np.ceil(df_ss['ss_95_raw']).fillna(0)

print("SKU Parametric Safety Stock Calculations (Head):")
display(df_ss[['sku_number', 'ss_75', 'ss_90', 'ss_95']].head())

SKU Parametric Safety Stock Calculations (Head):


,sku_number,ss_75,ss_90,ss_95
0,0019425F,69.0,131.0,168.0
1,00DCA10C,17.0,32.0,41.0
2,01FE860A,17.0,31.0,40.0
3,0336A033,74.0,139.0,179.0
4,058CD6C0,72.0,137.0,176.0


Assignment 2 Question 4 simply had us use `.nlargest()` to find the SKU with the highest safety stock requirement at 95% service level.

In [31]:
print("--- SKU with Largest Safety Stock (95% SL) ---")
sku_largest_ss = df_ss.nlargest(1, 'ss_95')
display(sku_largest_ss[['sku_number', 'ss_95']])

--- SKU with Largest Safety Stock (95% SL) ---


,sku_number,ss_95
89,528EC5AC,337.0


Assignment 2 Question 4 similarly had us use `.nsmallest()` to find the SKU with the lowest safety stock requirement at 95% service level.

In [32]:
print("--- SKU with Smallest Safety Stock (95% SL) ---")
sku_smallest_ss = df_ss.nsmallest(1, 'ss_95')
display(sku_smallest_ss[['sku_number', 'ss_95']])

--- SKU with Smallest Safety Stock (95% SL) ---


,sku_number,ss_95
176,9ED24ECA,32.0


Assignment 2 Question 6 had us now calculate the mean of `ss_95` to get the average safety stock across all SKUs.

In [33]:
avg_ss_95 = df_ss['ss_95'].mean()
print(f"\n--- Average Safety Stock (95% SL) ---")
print(f"The average safety stock (95% SL) across all SKUs is: {avg_ss_95:.2f} units")


--- Average Safety Stock (95% SL) ---
The average safety stock (95% SL) across all SKUs is: 105.29 units


Below is the 'Side-Quest' done in Assignment 2, with the possible motivation of extra credit.

A helper function is defined using `scipy.stats.mstats.mquantiles` to calculate empirical quantiles for a series of data.

In [34]:
def calc_mquantiles(series):
    probs = list(service_levels.values())
    quantiles = mstats.mquantiles(series, prob=probs)
    return pd.Series(quantiles, index=list(service_levels.keys()))

print("Helper function calc_mquantiles defined.")

Helper function calc_mquantiles defined.


The helper function is applied to `order_quantity` of every SKU group in the original `df_filtered` data.

In [35]:
print("Calculating empirical quantiles for each SKU's order_quantity...")

df_quantiles_np = df_filtered.groupby('sku_number')['order_quantity'].apply(calc_mquantiles).unstack()

df_quantiles_np = df_quantiles_np.rename(columns={
    '75%': 'q_np_75',
    '90%': 'q_np_90',
    '95%': 'q_np_95'
})

print("Quantile calculation complete. Head of quantiles:")
display(df_quantiles_np.head())

Calculating empirical quantiles for each SKU's order_quantity...


Quantile calculation complete. Head of quantiles:


,q_np_75,q_np_90,q_np_95
sku_number,,,
0019425F,112.0,124.0,131.00
00DCA10C,14.0,18.0,22.00
01FE860A,14.0,19.0,20.78
0336A033,114.0,128.0,135.39
058CD6C0,113.0,124.0,131.00


This cell constitutes the final steps of the side-quest, merging the new quantiles with `df_agg` and applying the non-parametric formula to find Safety Stock. We also use `.clip(lower=0)` to make sure safety stock cannot be negative.

In [36]:
df_ss_np = df_agg.merge(df_quantiles_np, on='sku_number')

df_ss_np['sqrt_lead_time'] = np.sqrt(df_ss_np['avg_lead_time'])

base_ss_75 = (df_ss_np['q_np_75'] - df_ss_np['qty_avg']).clip(lower=0)
base_ss_90 = (df_ss_np['q_np_90'] - df_ss_np['qty_avg']).clip(lower=0)
base_ss_95 = (df_ss_np['q_np_95'] - df_ss_np['qty_avg']).clip(lower=0)

df_ss_np['ss_np_75_raw'] = base_ss_75 * df_ss_np['sqrt_lead_time']
df_ss_np['ss_np_90_raw'] = base_ss_90 * df_ss_np['sqrt_lead_time']
df_ss_np['ss_np_95_raw'] = base_ss_95 * df_ss_np['sqrt_lead_time']

df_ss_np['ss_np_75'] = np.ceil(df_ss_np['ss_np_75_raw']).fillna(0)
df_ss_np['ss_np_90'] = np.ceil(df_ss_np['ss_np_90_raw']).fillna(0)
df_ss_np['ss_np_95'] = np.ceil(df_ss_np['ss_np_95_raw']).fillna(0)

print("\nNon-Parametric Safety Stock Calculations (Head):")
display(df_ss_np[['sku_number', 'ss_np_75', 'ss_np_90', 'ss_np_95']].head())


Non-Parametric Safety Stock Calculations (Head):


,sku_number,ss_np_75,ss_np_90,ss_np_95
0,0019425F,71.0,134.0,171.0
1,00DCA10C,16.0,31.0,46.0
2,01FE860A,17.0,36.0,42.0
3,0336A033,73.0,147.0,186.0
4,058CD6C0,76.0,135.0,172.0


This section lists all packages and key functions used in this analysis.

Packages: `pandas`, `numpy`, `scipy.stats.norm`, `scipy.stats.mstats`

Key Functions:

`pandas`: `.read_csv()`, `.rename()`, `.dropna()`, `.describe()`, `.nunique()`, `.nlargest()`, `.nsmallest()`, `.groupby()`, `.agg()`, `.copy()`, `.merge()`, `.apply()`, `.unstack()`, `.clip()`, `.fillna()`

`numpy`: `np.sqrt()`, `np.ceil()`

`scipy`: `norm.ppf()`, `mstats.mquantiles()`

No other external sources were used.